# 文本遮掩稳健性测试（Y1／Y2）

目的：在**冻结测试集**上，把原文的一部分汉字替换为 `□`，再用同一套冻结模型评分，检验 Y 分数对轻度文本缺失是否稳定。

这不是重新训练，也不改动原始数据、标签、X、ENV 或关联分析。遮掩位置由固定随机种子和 `row_id` 决定，因此可复现。

In [ ]:
from pathlib import Path
import sys, random
import numpy as np
import pandas as pd
from scipy.special import expit
from scipy.stats import spearmanr
from threadpoolctl import threadpool_limits

PYTHON_DIR = Path.cwd() if (Path.cwd() / 'analysis_helpers.py').exists() else Path.cwd() / 'python'
if not (PYTHON_DIR / 'analysis_helpers.py').exists():
    raise FileNotFoundError('请从 lasso-1016 或 lasso-1016/python 目录打开本 Notebook。')
sys.path.insert(0, str(PYTHON_DIR.resolve()))
import analysis_helpers as h
if h.ROOT.resolve() != PYTHON_DIR.resolve().parent:
    raise RuntimeError('当前内核已加载另一个项目版本；请重启内核后重试。')
print(h.runtime_check())

MASK_RATE = 0.20       # 遮掩 20% 汉字；可改为 0.10、0.30 等
SEED = 20261017        # 固定后不要改，保证每次遮掩位置一致
MASK_TOKEN = '□'
POSITIVE_THRESHOLD = 0.50
assert 0 < MASK_RATE < 1


In [ ]:
def is_han(ch):
    return '\u4e00' <= ch <= '\u9fff'

def mask_text(text, row_id, rate=MASK_RATE, seed=SEED):
    """只遮掩汉字；保留数字、标点和网址结构。"""
    text = '' if pd.isna(text) else str(text)
    candidates = [i for i, ch in enumerate(text) if is_han(ch)]
    if not candidates:
        return text
    n_mask = max(1, round(len(candidates) * rate))
    rng = random.Random(f'{seed}::{row_id}')
    chosen = set(rng.sample(candidates, n_mask))
    return ''.join(MASK_TOKEN if i in chosen else ch for i, ch in enumerate(text))

def score_texts(name, texts):
    vectorizer, coef, intercept = h.load_scoring_model(name)
    with threadpool_limits(limits=1):
        matrix = vectorizer.transform(texts)
        return np.asarray(expit(matrix @ coef + intercept)).ravel()


In [ ]:
def run_mask_test(name):
    pool, plan = h.load_target(name)
    test = pool.set_index('row_id').loc[plan['test_ids']].reset_index().copy()
    # 基线也由原文重新评分；不读取关联分析中的旧分数。
    test['score_original'] = score_texts(name, test['text'].tolist())
    test['text_masked'] = [mask_text(t, rid) for rid, t in zip(test['row_id'], test['text'])]
    test['score_masked'] = score_texts(name, test['text_masked'].tolist())
    test['abs_change'] = (test['score_masked'] - test['score_original']).abs()
    test['same_side_of_0_5'] = (test['score_original'] >= POSITIVE_THRESHOLD) == (test['score_masked'] >= POSITIVE_THRESHOLD)
    rho, _ = spearmanr(test['score_original'], test['score_masked'])
    summary = {
        '目标': name,
        '冻结测试集条数': len(test),
        '遮掩比例': MASK_RATE,
        'Spearman相关': float(rho),
        '平均绝对变化(MAE)': float(test['abs_change'].mean()),
        '95%绝对变化分位数': float(test['abs_change'].quantile(.95)),
        '0.5阈值侧别一致率': float(test['same_side_of_0_5'].mean()),
        '原文均分': float(test['score_original'].mean()),
        '遮掩后均分': float(test['score_masked'].mean()),
    }
    return test, summary

details, summaries = {}, []
for NAME in ('Y1', 'Y2'):
    details[NAME], summary = run_mask_test(NAME)
    summaries.append(summary)
summary_df = pd.DataFrame(summaries)
display(summary_df)


In [ ]:
# 人工查看变化最大的例子；这一步只用于解释模型对哪些文本敏感，不删除或改写任何记录。
for NAME, frame in details.items():
    print(f'\n{NAME}：绝对变化最大的 10 条')
    display(frame.nlargest(10, 'abs_change')[['row_id', 'text', 'text_masked', 'score_original', 'score_masked', 'abs_change']])

# 如确有保存需要，再手动解除下一行注释；默认不写文件。
# summary_df.to_csv(h.DATA / '06_1016工作数据' / '文本遮掩稳健性汇总.csv', index=False, encoding='utf-8-sig')
